# Advanced genetic architectures

Here we consider more complicated genetic-architecture configurations
that go beyond the standard one-trait infinitesimal model.

## Partially overlapping non-infinitesimal models

Consider effects for two traits $Y$ and $Z$ where the genome is split
into five disjoint variant sets:

1. variants causal for $Y$ only
2. variants causal for $Z$ only
3. variants causal for both with **orthogonal** effects
4. variants causal for both with **correlated** effects ($r_β = 0.5$)
5. non-causal variants

We'll set $h^2_Y = 0.5$ and $h^2_Z = 0.4$, with each causal set
contributing one third of the total heritability.

The simplest way to set this up is to build the per-variant effect
matrix by hand and wrap it with `xftsim.effect.MultivariateEffects.from_array`:


In [ ]:
import numpy as np
import xftsim as xft

np.random.seed(123)

h2_y, h2_z = 0.5, 0.4
r_yz = 0.5

N, M = 4000, 1000
hap = xft.founders.founder_haplotypes_uniform_AFs(n=N, m=M)

# Partition variants into five equally sized sets.
variant_sets = [np.sort(x) for x in np.array_split(np.random.permutation(M), 5)]


In [ ]:
beta = np.zeros((M, 2))

# 1. Y-only
idx = variant_sets[0]
beta[idx, 0] = np.random.randn(len(idx)) * np.sqrt(h2_y / 3 / len(idx))

# 2. Z-only
idx = variant_sets[1]
beta[idx, 1] = np.random.randn(len(idx)) * np.sqrt(h2_z / 3 / len(idx))

# 3. Y & Z with orthogonal effects
idx = variant_sets[2]
beta[idx, 0] = np.random.randn(len(idx)) * np.sqrt(h2_y / 3 / len(idx))
beta[idx, 1] = np.random.randn(len(idx)) * np.sqrt(h2_z / 3 / len(idx))

# 4. Y & Z with correlated effects (r = 0.5)
idx = variant_sets[3]
cov = np.array([[h2_y / 3, r_yz * np.sqrt(h2_y * h2_z) / 3],
                [r_yz * np.sqrt(h2_y * h2_z) / 3, h2_z / 3]]) / len(idx)
beta[idx, :] = np.random.multivariate_normal(np.zeros(2), cov, size=len(idx))

# 5. non-causal: already zero


Wrap the effect matrix in a `MultivariateEffects` and reference it
from the formula via the `mvGenetic` builtin:


In [ ]:
eff_yz = xft.effect.MultivariateEffects.from_array(beta, standardized=True)

arch = xft.arch.Architecture(
    formula=f'''
    (y.G, z.G) ~ mvGenetic(eff_yz)
    y.E ~ noise({1 - h2_y})
    z.E ~ noise({1 - h2_z})
    y ~ y.G + y.E
    z ~ z.G + z.E
    ''',
    effects={'eff_yz': eff_yz},
)


Verify the genetic covariance structure on standardised genotypes:


In [ ]:
G_std = hap.to_diploid_standardized(scale=True)
print('Total genetic covariance:')
print(np.cov(G_std @ beta, rowvar=False))
print()
for i, idx in enumerate(variant_sets):
    print(f'Set {i} genetic covariance:')
    print(np.cov(G_std[:, idx] @ beta[idx, :], rowvar=False))
    print()


## Running the simulation under cross-trait assortative mating

Under bivariate cross-mate cross-trait assortment, even with truly
orthogonal genetic effects across sets, the HE-estimated genetic
correlation will increase generation by generation. We assume an
exchangeable cross-mate correlation of $r_{mate} = 0.5$ on the two
phenotypes:


In [ ]:
rmap = xft.reproduce.RecombinationMap.from_haplotypes(hap, p=0.1)
mating = xft.mate.LinearAssortativeMating(
    component_names=['y', 'z'], r=0.5,
)

sim = xft.sim.Simulation(
    founder_haplotypes=hap, architecture=arch, recombination_map=rmap,
    mating_regime=mating,
    statistics=[xft.stats.SampleStatistics(),
                xft.stats.MatingStatistics(),
                xft.stats.HasemanElstonEstimator(phenotype_keys=['y', 'z'])],
    filters={'trio': xft.filters.TrioFilter()},
    seed=42,
)
sim.run(n_generations=5)


We can read out the genetic-covariance estimate and per-generation HE
correlation between `y` and `z`:


In [ ]:
for r in sim.results:
    he = r.statistics.get('HasemanElstonEstimator', {})
    cov_g = he.get('_cov_g')
    if cov_g is None:
        continue
    d = np.sqrt(np.abs(np.diag(cov_g)))
    d[d < 1e-15] = 1.0
    rg = float(cov_g[0, 1] / (d[0] * d[1]))
    print(f'gen {r.generation}: HE rg(y, z) = {rg:.3f}')


We can also write the haplotypes out to PLINK1 if we want to run
external analyses on the final generation:


In [ ]:
# xft.io.write_to_plink1 doesn't exist in v0.9 — the npz round-trip is
# the recommended path. See `xftsim.io.save_haplotypes_npz` and
# `load_haplotypes_npz`.
final_hap = sim.haplotype_history[sim.generation]
# xft.io.save_haplotypes_npz(final_hap, '/tmp/final_hap.npz')
